In [505]:
import os
import math
import matplotlib.pyplot as plt

import verde as vd
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, box



In [506]:
_dir = '/home/jan/Documents/EPFL/project_2/data/labels/dataset_v5'

dfs = []
#sample_dst = []
for _file in os.listdir(_dir):
    if not _file.endswith('.csv'):
        continue
    if _file in ['labels.csv', 'labels_v0', 'cocoa_th_65.csv']:
        continue
    if _file.startswith('labels'):
        continue
    print(_file)
    df = pd.read_csv(os.path.join(_dir, _file))    
    
    df['dataset'] = _file
    
    
    '''try:
        dst_samples = df.sample(100, random_state=1)
    except ValueError:
        dst_samples = df
        print(dst_samples.shape)
    dst_samples['dataset'] = _file
    sample_dst.append(dst_samples)'''

    dfs.append(df)


cocoa_peru_points.csv
manual_annotation.csv
mapbiomas_bolivia.csv
africa_masolele.csv
colombia_pasture.csv
mapbiomas_paraguay.csv
rubber.csv
cocoa.csv
mapbiomas.csv
vancutsem_regrowth.csv
coffee_maskell.csv
mapbiomas_indonesia.csv
palm_oil.csv
worldcover.csv
coffee_sdpt_brazil.csv
cocoa_ecuador.csv
tyukavina_congo_basin_agri.csv
mines.csv
soy_south_america.csv
mapbiomas_peru.csv
mapbiomas_venezuela.csv


In [507]:
df = pd.concat(dfs)

In [508]:
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])


In [509]:
gdf = gpd.GeoDataFrame(df)
gdf['geometry'] = gdf.geometry.centroid

In [510]:
gdf = gdf.reset_index(drop=True)

In [511]:
gdf.driver.value_counts()

driver
pasture                   694080
palm oil                  391368
soy                       158671
forest plantation          61818
rubber                     50469
forest regrowth            49325
rice                       42746
mining                     42025
urban                      23858
shifting agriculture       17360
river, lake, ocean         16229
cocoa                      16041
other temporary crops       3333
sugar cane                  1432
coffee                       806
rocky outcrop                778
beach, dune, snad            464
hypersaline tidal flat       134
aquaculture                   44
citrus                        42
cotton                        37
oil palm                      13
water                          4
Name: count, dtype: int64

In [512]:
#Add country name
world_filepath = gpd.datasets.get_path('naturalearth_lowres')
world = gpd.read_file(world_filepath)

min_lon = -180
max_lon = 180
min_lat = -30
max_lat = 30
tropics = Polygon([(min_lon, min_lat), (min_lon, max_lat), (max_lon, max_lat, ), (max_lon, min_lat )])
tropical_countries = world.loc[world.intersects(tropics)]



gdf.crs = 4326
gdf = gpd.sjoin(gdf, tropical_countries[['geometry', 'name']], how='left', predicate='within')[['geometry', 'driver', 'sampleid', 'dataset', 'name']]
gdf['country'] = gdf['name']
gdf = gdf.drop('name', axis=1)

/tmp/ipykernel_2854489/54163365.py:2: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world_filepath = gpd.datasets.get_path('naturalearth_lowres')


In [513]:
gdf['driver'] = gdf['driver'].replace({
    'river, lake, ocean': 'other', 
    'other temporary crops': 'other',
    'aquaculture': 'other',
    'sugar cane': 'other',
    'beach, dune, snad': 'other',
    'rocky outcrop': 'other',
    'hypersaline tidal flat': 'other',
     'rice': 'other',
    'cotton': 'other',
    'citrus': 'other',
    'water': 'other',
    'oil palm' : 'palm oil'
})


In [514]:
gdf.driver.value_counts()

driver
pasture                 694080
palm oil                391381
soy                     158671
other                    65243
forest plantation        61818
rubber                   50469
forest regrowth          49325
mining                   42025
urban                    23858
shifting agriculture     17360
cocoa                    16041
coffee                     806
Name: count, dtype: int64

In [515]:
gdf.driver.value_counts()

driver
pasture                 694080
palm oil                391381
soy                     158671
other                    65243
forest plantation        61818
rubber                   50469
forest regrowth          49325
mining                   42025
urban                    23858
shifting agriculture     17360
cocoa                    16041
coffee                     806
Name: count, dtype: int64

In [516]:
gdf.loc[gdf.driver == 'cocoa'].country.value_counts()

country
Ghana               5536
Ecuador             5510
Côte d'Ivoire       4857
Peru                  59
Papua New Guinea      48
Colombia              15
Cameroon              10
Venezuela              2
Name: count, dtype: int64

In [517]:

gdf.loc[gdf.dataset == 'manual_annotation.csv'].shape

(405, 5)

## Overlapping samples

- identify where examples from different datasets overlap with each other
- disregard those where the labels are the same - that's good news
- look at the rest

In [518]:
gdf.crs = 4326

gdf['geometry'] = gdf.to_crs(3857).geometry.buffer(500, cap_style=3).to_crs(4326)
# Perform spatial join to find intersecting polygons
# This creates a new GeoDataFrame where each row represents pairs of intersecting polygons
intersections = gpd.sjoin(gdf, gdf, how="inner", op='intersects')

# intersections will contain all pairs of polygons that intersect
# Note: this includes each polygon intersecting with itself, so you may want to remove those entries
intersections = intersections[intersections.index != intersections.index_right]
#intersections.to_file('../data/tmp/tmp.gpkg')

/home/jan/miniconda3/envs/torch/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3466: FutureWarning: The `op` parameter is deprecated and will be removed in a future release. Please use the `predicate` parameter instead.
  if await self.run_code(code, result, async_=asy):


In [519]:
overlap_label_conflict = intersections.loc[~(intersections.driver_left == intersections.driver_right)]

In [520]:
#TODO:
# in case of conflict (two examples with different labels), i remove both
# add manually examples from manual_annotation because these should be correct
conflict_without_manual = overlap_label_conflict.loc[overlap_label_conflict.dataset_left != 'manual_annotation.csv']

In [521]:
gdf_without_conflicts = gdf[~gdf.index.isin(conflict_without_manual.index)].reset_index(drop=True)

In [522]:
disagreeing_overlaps['class_mismatch'] = disagreeing_overlaps.driver_left + ", " + disagreeing_overlaps.driver_right
disagreeing_overlaps.class_mismatch.value_counts().head(9)



class_mismatch
soy, pasture                   37813
pasture, soy                   37813
forest plantation, rubber      14715
rubber, forest plantation      14715
forest regrowth, pasture       12199
pasture, forest regrowth       12199
palm oil, forest plantation    11438
forest plantation, palm oil    11438
rubber, palm oil               10861
Name: count, dtype: int64

In [523]:
disagreeing_overlaps.to_file('../data/tmp/conflicts_dst_v5.gpkg')

In [524]:
pd.DataFrame(gdf_without_conflicts).to_csv('../data/labels/dataset_v5/labels_v0.csv', index=False)

In [525]:
gdf_without_conflicts.dataset.value_counts()

dataset
mapbiomas.csv                     655494
mapbiomas_indonesia.csv           241382
palm_oil.csv                      180703
soy_south_america.csv              87145
colombia_pasture.csv               63692
rubber.csv                         43157
vancutsem_regrowth.csv             39947
mines.csv                          30250
mapbiomas_paraguay.csv             22813
worldcover.csv                     18000
tyukavina_congo_basin_agri.csv     17002
mapbiomas_bolivia.csv              12003
cocoa.csv                          10144
mapbiomas_peru.csv                  8678
cocoa_ecuador.csv                   5011
mapbiomas_venezuela.csv             1945
manual_annotation.csv                405
coffee_maskell.csv                   283
africa_masolele.csv                  164
cocoa_peru_points.csv                 44
coffee_sdpt_brazil.csv                41
Name: count, dtype: int64

In [561]:
gdf_without_conflicts.loc[gdf_without_conflicts.driver == 'cocoa'].country.value_counts()

country
Ghana               5307
Ecuador             5005
Côte d'Ivoire       4837
Papua New Guinea      48
Peru                  45
Colombia              15
Cameroon              10
Venezuela              2
Name: count, dtype: int64

### Sampling strategy


1. first check if all datasets have some examples
2. check if the number of examples from some datasets is suspicious (too low?)
3. select X number of smallest datasets and add them to the final dst
4. 

In [526]:
#mapb = gdf_without_conflicts.loc[gdf_without_conflicts.dataset == 'mapbiomas.csv']
#mapb.driver.value_counts()
#gdf_without_conflicts.loc[gdf_without_conflicts.driver == 'pasture']['country'].value_counts()

## Sample a diverse dataset

### Find n smallest datasets, add all of them 


In [562]:
#Identify smallest datasets
value_counts = gdf_without_conflicts['dataset'].value_counts()

dataset_sizes = value_counts.reset_index()
dataset_sizes.columns = ['value', 'count']

dataset_sizes = dataset_sizes.sort_values(by='count').reset_index(drop=True)
dataset_sizes



,value,count
0,coffee_sdpt_brazil.csv,41
1,cocoa_peru_points.csv,44
2,africa_masolele.csv,164
3,coffee_maskell.csv,283
4,manual_annotation.csv,405
5,mapbiomas_venezuela.csv,1945
6,cocoa_ecuador.csv,5011
7,mapbiomas_peru.csv,8678
8,cocoa.csv,10144
9,mapbiomas_bolivia.csv,12003


In [563]:
#Get all examples from n_smallest datasets
n = 5
n_smallest_datasets = dataset_sizes.iloc[:n].value.tolist()

small_datasets_examples = gdf_without_conflicts.loc[gdf_without_conflicts.dataset.isin(n_smallest_datasets)] 

In [564]:
#For the rest, using those examples that have not already been selected
filtered_gdf = gdf_without_conflicts.loc[~gdf_without_conflicts.dataset.isin(n_smallest_datasets)] 

### Identify non-commodity classes

In [565]:
commodity_classes = ['cocoa', 'coffee', 'rubber',  'pasture', 
        'forest plantation', 'palm oil', 'soy']

non_commodity_classes = [ 'other', 'urban', 'mining', 'shifting agriculture', 'forest regrowth']
non_commodity_gdf = filtered_gdf.loc[filtered_gdf.driver.isin(non_commodity_classes)]

In [566]:
def get_grid(size, latitude_range=(-30, 30), longitude_range=(-180, 180), select_countries=None):

    # Load the natural earth dataset of the world
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    
    if select_countries:
        world = world.loc[world.name.isin(select_countries)]


    # Create a list to hold the polygons
    polygons = []
    

    # Generate the grid
    for lat in range(latitude_range[0], latitude_range[1], size):
        for lon in range(longitude_range[0], longitude_range[1], size):
            # Define the coordinates of the polygon
            coords = [
                (lon, lat),
                (lon + size, lat),
                (lon + size, lat + size),
                (lon, lat + size),
                (lon, lat)
            ]
            # Create a polygon from the coordinates
            polygon = Polygon(coords)
            # Check if the polygon intersects with any land area
            if world.intersects(polygon).any():
                polygons.append(polygon)

    # Create a GeoDataFrame from the list of polygons
    grid = gpd.GeoDataFrame({'geometry': polygons})

    # Set the coordinate reference system (CRS) to WGS84
    _ = grid.set_crs(epsg=4326, inplace=True)
    
    return grid

#gdf

#gdf.to_file('../data/tmp/delete.gpkg')

In [567]:
def sample_dataframe(df, num_samples, seed=1):
    if len(df) <= num_samples:
        return df
    else:
        return df.sample(n=num_samples, random_state=seed)

In [568]:
grid = get_grid(5)
samples = []
max_per_cell = 10
for _cls in non_commodity_classes:
    
    _subset_gdf = gdf_without_conflicts.loc[gdf_without_conflicts.driver == _cls]
    
    for i, row in grid.iterrows():
        geom = row.geometry
        cell_subset = _subset_gdf.loc[_subset_gdf.intersects(geom)]
        if cell_subset.empty:
        
            continue
        samples.append(sample_dataframe(cell_subset, max_per_cell))


/tmp/ipykernel_2854489/559284933.py:4: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))


In [534]:
stratified_samples = gpd.GeoDataFrame(pd.concat(samples).reset_index(drop=True))
#stratified_samples.to_file('../data/tmp/test.gpkg')

In [469]:
stratified_samples.driver.value_counts()

driver
forest regrowth         1738
urban                   1205
mining                  1147
other                    708
shifting agriculture     257
Name: count, dtype: int64

In [535]:
stratified_samples.driver.value_counts()

driver
forest regrowth         1738
urban                   1205
mining                  1147
other                    708
shifting agriculture     257
Name: count, dtype: int64

In [536]:
# Exclude the rows i sampled above
mask = non_commodity_gdf.index.isin(pd.concat(samples).index)
filtered_non_commodity = non_commodity_gdf[~mask]

In [537]:
non_commodity_classes

['other', 'urban', 'mining', 'shifting agriculture', 'forest regrowth']

In [538]:
samples_2 = []
total_samples_per_class = 3000
for _cls in non_commodity_classes:
    
    if _cls == 'shifting agriculture':
            samples_per_class = total_samples_per_class - stratified_samples.driver.value_counts()[_cls] - \
                small_datasets_examples.driver.value_counts()['shifting agriculture']
    else:
        samples_per_class = total_samples_per_class - stratified_samples.driver.value_counts()[_cls]
    
    _subset_gdf = filtered_non_commodity.loc[filtered_non_commodity.driver == _cls]

    samples_2.append(sample_dataframe(_subset_gdf, samples_per_class))


In [539]:
random_samples = gpd.GeoDataFrame(pd.concat(samples_2).reset_index(drop=True))



In [540]:
non_commodity_samples = gpd.GeoDataFrame(pd.concat([stratified_samples, random_samples]).reset_index(drop=True))


In [541]:
non_commodity_samples.driver.value_counts()

driver
other                   3000
urban                   3000
mining                  3000
forest regrowth         3000
shifting agriculture    2873
Name: count, dtype: int64

In [476]:
#non_commodity_samples.to_file('../data/tmp/tmp_non_commodity.gpkg')

In [542]:
non_commodity_samples['geometry'] = non_commodity_samples['geometry'].centroid
non_commodity_samples.to_file('../data/tmp/tmp_non_commodity.gpkg')

/tmp/ipykernel_2854489/2470950196.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  non_commodity_samples['geometry'] = non_commodity_samples['geometry'].centroid


## Commodity-related class sampling

### load the deduce stats

In [543]:

deduce = pd.read_csv("/home/jan/Documents/EPFL/project_2/data/tabular/deduce_yearly.csv")
per_country = deduce.groupby(['country', 'Commodity']).sum().reset_index()
per_country = per_country.pivot(index='country', columns='Commodity', values='commodity attributed deforestation')

def sort_by_column_ratio(df, column):
    """Sort the dataframe by the ratio of the specified column to the sum of all columns.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing numerical values or NaNs.
    column (str): The column to calculate the ratio for.
    
    Returns:
    pd.DataFrame: DataFrame sorted by the ratio of the specified column to the sum of all columns.
    """
    # Replace NaNs with 0 for sum calculation
    df_filled = df.fillna(0)
    
    # Calculate the sum of all columns for each row
    row_sum = df_filled.sum(axis=1)
    
    # Calculate the ratio of the specified column to the sum of all columns
    df['ratio'] = df_filled[column] / row_sum
    
    # Sort the dataframe by the ratio in descending order
    sorted_df = df.sort_values(by='ratio', ascending=False).drop(columns='ratio')
    
    return sorted_df



In [479]:
per_country.columns

Index(['Cattle meat', 'Cocoa beans', 'Coffee, green',
       'Forest plantation (Alder)', 'Forest plantation (Australian Blackwood)',
       'Forest plantation (Bamboo)', 'Forest plantation (Birch)',
       'Forest plantation (Broadleaf)', 'Forest plantation (Fir)',
       'Forest plantation (Mix of needleleaf and broadleaf)',
       'Forest plantation (Needleleaf)', 'Forest plantation (Padauk)',
       'Forest plantation (Pine)', 'Forest plantation (Sal)',
       'Forest plantation (Teak)', 'Forest plantation (Unclassified)',
       'Leather', 'Natural rubber in primary forms', 'Oil palm fruit',
       'Soya beans'],
      dtype='object', name='Commodity')

In [480]:
#this cell i dont really need


#TODO: filter out the small datasets that i have already sampled from? or just at the end, remove duplicates

#commodity_classes = ['cocoa', 'coffee', 'rubber',  'pasture', 
#        'forest plantation', 'palm oil', 'soy']

#commodity_gdf = filtered_gdf.loc[filtered_gdf.driver.isin(commodity_classes)]

In [544]:
commodity_gdf.driver.value_counts()

driver
pasture              1319172
palm oil              733496
soy                   267374
forest plantation      94990
rubber                 86348
cocoa                  30426
coffee                  1319
Name: count, dtype: int64

### Soy

In [545]:
_cls = 'soy'
_subset_gdf = filtered_gdf.loc[filtered_gdf.driver == _cls]
print(len(_subset_gdf))


133688


In [546]:
_subset_gdf.country.value_counts()

country
Brazil       109873
Paraguay      12441
Bolivia       11228
Argentina       145
Name: count, dtype: int64

In [548]:
sort_by_column_ratio(per_country, 'Soya beans').head(5)

Commodity,Cattle meat,Cocoa beans,"Coffee, green",Forest plantation (Alder),Forest plantation (Australian Blackwood),Forest plantation (Bamboo),Forest plantation (Birch),Forest plantation (Broadleaf),Forest plantation (Fir),Forest plantation (Mix of needleleaf and broadleaf),Forest plantation (Needleleaf),Forest plantation (Padauk),Forest plantation (Pine),Forest plantation (Sal),Forest plantation (Teak),Forest plantation (Unclassified),Leather,Natural rubber in primary forms,Oil palm fruit,Soya beans
country,,,,,,,,,,,,,,,,,,,,
Belize,3.340203,1.290443,0.043566,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.488799,0.175800,NaN,0.006985,15.852755
Rwanda,NaN,NaN,1.733621,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000244,NaN,NaN,0.658416,NaN,NaN,0.553183,4.339003
Zambia,15.700090,NaN,0.069195,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.826321,NaN,NaN,13.276410
Benin,12.634286,NaN,0.000910,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.034094,0.664962,NaN,1.723134,7.433725
Bolivia (Plurinational State of),47.307818,0.095735,0.165045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000029,2.489885,NaN,NaN,16.963033


In [549]:
### Argentina is not included in the tabular data
### all other three countries are very high in the tab data for soy production -> good
_subset_gdf = _subset_gdf.loc[_subset_gdf.country != 'Argentina'].reset_index(drop=True)

In [550]:
grid_1 = get_grid(1, latitude_range=(-30, 6), longitude_range=(-70, -20))

soy_samples = []
max_per_cell = 10 #Here the number is higher since it's only 3 countries
for i, row in grid_1.iterrows():
    geom = row.geometry
    cell_subset = _subset_gdf.loc[_subset_gdf.intersects(geom)]
    if cell_subset.empty:
        continue
    soy_samples.append(sample_dataframe(cell_subset, max_per_cell))


/tmp/ipykernel_2854489/559284933.py:4: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))


In [551]:
soy_stratified_samples = gpd.GeoDataFrame(pd.concat(soy_samples))


In [552]:
soy_stratified_samples.country.value_counts()

country
Brazil      2431
Paraguay     158
Bolivia      131
Name: count, dtype: int64

In [553]:

# Exclude the rows i sampled above
mask = _subset_gdf.index.isin(soy_stratified_samples.index)
filtered_soy = _subset_gdf[~mask]


In [554]:
filtered_soy.shape

(130838, 5)

In [556]:

total_samples_per_class = 3000

n_random_samples_soy = total_samples_per_class - len(soy_stratified_samples)

random_samples_soy = sample_dataframe(filtered_soy, n_random_samples_soy)


In [557]:
soy_samples = gpd.GeoDataFrame(pd.concat([soy_stratified_samples, random_samples_soy]).reset_index(drop=True))


In [558]:
soy_samples.to_file('../data/tmp/test_soy.gpkg')

### Cocoa

strategy:
- sample all from minor countries - this is already done
- sample over grid
- sample extra from Ghana + Cote d'Ivoire
- sample rest randomly

In [493]:
_cls = 'cocoa'
_cocoa_subset_gdf = filtered_gdf.loc[filtered_gdf.driver == _cls]
print(len(_cocoa_subset_gdf))
_cocoa_subset_gdf.country.value_counts()

133543


country
Ghana            5307
Ecuador          5005
Côte d'Ivoire    4837
Colombia            3
Peru                1
Name: count, dtype: int64

In [570]:
select_countries = ['Ghana', "Côte d'Ivoire", 'Ecuador', 'Peru', 'Colombia']

In [573]:
grid_cocoa = get_grid(1, select_countries=select_countries)

cocoa_samples = []
max_per_cell = 50
for i, row in grid_cocoa.iterrows():
    geom = row.geometry
    cell_subset = _cocoa_subset_gdf.loc[_cocoa_subset_gdf.intersects(geom)]
    if cell_subset.empty:
        continue
    cocoa_samples.append(sample_dataframe(cell_subset, max_per_cell))

cocoa_stratified_samples = gpd.GeoDataFrame(pd.concat(cocoa_samples))


/tmp/ipykernel_2854489/559284933.py:4: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))


In [574]:
cocoa_stratified_samples.country.value_counts()

country
Côte d'Ivoire    660
Ecuador          602
Ghana            442
Colombia           3
Peru               1
Name: count, dtype: int64

In [575]:

# Exclude the rows i sampled above
cocoa_mask = _cocoa_subset_gdf.index.isin(cocoa_stratified_samples.index)
filtered_cocoa = _cocoa_subset_gdf[~cocoa_mask]

total_samples_per_class = 3000


In [ ]:
#Here I will sample more cocoa examples from the countries where it makes more sense, ie. less from Ecuador

In [577]:
west_africa_samples = sample_dataframe(filtered_cocoa.loc[filtered_cocoa.country.isin(["Côte d'Ivoire", "Ghana"])], 1000)



In [579]:
cocoa_samples = gpd.GeoDataFrame(pd.concat([cocoa_stratified_samples, west_africa_samples]))


In [581]:
cocoa_samples.country.value_counts()

country
Côte d'Ivoire    1146
Ghana             956
Ecuador           602
Colombia            3
Peru                1
Name: count, dtype: int64

In [583]:

n_random_samples_soy = total_samples_per_class - len(cocoa_samples)
print(n_random_samples_soy)


random_samples_soy = sample_dataframe(filtered_soy, n_random_samples_soy)


290


In [584]:
cocoa_samples = gpd.GeoDataFrame(pd.concat([cocoa_samples, random_samples_soy]))


### Coffee

strategy:
- sample all

In [586]:
_cls = 'coffee'
_coffee_subset_gdf = filtered_gdf.loc[filtered_gdf.driver == _cls]
_coffee_subset_gdf.country.value_counts()



123


country
Brazil    123
Name: count, dtype: int64

In [ ]:
coffee_samples = _coffee_subset_gdf

### Pasture

strategy:




In [588]:
_cls = 'pasture'
_pasture_subset_gdf = filtered_gdf.loc[filtered_gdf.driver == _cls]
_pasture_subset_gdf.country.value_counts()



country
Brazil       558756
Colombia      63596
Paraguay      22744
Bolivia       10258
Venezuela      2032
Peru           1898
Argentina        76
Guyana            5
Ecuador           4
France            3
Panama            1
Name: count, dtype: int64

In [592]:
sort_by_column_ratio(per_country, 'Cattle meat').head(30)

Commodity,Cattle meat,Cocoa beans,"Coffee, green",Forest plantation (Alder),Forest plantation (Australian Blackwood),Forest plantation (Bamboo),Forest plantation (Birch),Forest plantation (Broadleaf),Forest plantation (Fir),Forest plantation (Mix of needleleaf and broadleaf),Forest plantation (Needleleaf),Forest plantation (Padauk),Forest plantation (Pine),Forest plantation (Sal),Forest plantation (Teak),Forest plantation (Unclassified),Leather,Natural rubber in primary forms,Oil palm fruit,Soya beans
country,,,,,,,,,,,,,,,,,,,,
Bahamas,55.548494,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.923605,NaN,NaN,NaN
South Sudan,35.342954,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.026679,1.860155,NaN,NaN,NaN
Guyana,41.724920,0.083969,0.208813,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.196048,NaN,NaN,NaN
Suriname,45.939985,0.028635,0.268726,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.417894,NaN,0.132449,0.015044
Kenya,24.993111,NaN,0.212332,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002634,1.315427,NaN,NaN,0.039195
Madagascar,60.783823,0.303919,1.341349,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.444510,3.199149,NaN,0.002069,0.000097
Mozambique,36.749842,NaN,0.002597,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.168185,1.934202,NaN,NaN,1.304519
Nicaragua,79.866217,0.241116,1.541162,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.611724,4.203485,NaN,1.441899,0.161148
France,31.125288,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.638173,NaN,NaN,1.959915


In [590]:
per_country

Commodity,Cattle meat,Cocoa beans,"Coffee, green",Forest plantation (Alder),Forest plantation (Australian Blackwood),Forest plantation (Bamboo),Forest plantation (Birch),Forest plantation (Broadleaf),Forest plantation (Fir),Forest plantation (Mix of needleleaf and broadleaf),...,Forest plantation (Padauk),Forest plantation (Pine),Forest plantation (Sal),Forest plantation (Teak),Forest plantation (Unclassified),Leather,Natural rubber in primary forms,Oil palm fruit,Soya beans,ratio
country,,,,,,,,,,,,,,,,,,,,,
Angola,28.713242,0.012401,0.883636,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.511223,NaN,0.013145,1.097869,0.034026
Australia,13.507555,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,62.590021,0.710924,NaN,NaN,0.071757,0.000933
Bahamas,55.548494,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,2.923605,NaN,NaN,NaN,0.000000
Bangladesh,15.621705,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,11.943748,0.822195,2.409705,NaN,0.567638,0.018087
Belize,3.340203,1.290443,0.043566,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.488799,0.175800,NaN,0.006985,15.852755,0.722341
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
United States of America,6.418858,NaN,0.000355,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,82.170583,0.337835,NaN,NaN,3.040608,0.033050
Vanuatu,19.730972,24.193539,0.350354,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,1.038472,NaN,0.016324,NaN,0.000000
Venezuela (Bolivarian Republic of),47.447586,1.519243,0.135711,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,17.698015,2.497241,NaN,1.147772,1.452792,0.020201


## OLD Sampling limited number of examples per class

 - first, do it just by simply randomly sampling
 - look at the results in QGIS and check spatial bias (e.g., if 

In [4]:
gdf = pd.read_csv('../data/labels/dataset_v4/labels.csv')

In [335]:
gdf = gdf_without_conflicts 

In [336]:
driver_classes = np.unique(gdf.driver, return_counts=True)[0]
max_count_per_class = 5000
def flatten(nested_list):
    return [x for xs in nested_list for x in xs]


indices = []
for _class in driver_classes:
    try:
        class_samples = gdf.loc[gdf.driver == _class].sample(max_count_per_class, random_state=1).index
    except ValueError:
        class_samples = gdf.loc[gdf.driver == _class].index
    indices.append(class_samples.tolist())

indices = flatten(indices)

subset = gdf.loc[indices]

In [337]:
cls, count = np.unique(subset.driver, return_counts=True)


for i in range(len(cls)):
    print(cls[i], count[i])

cocoa 5000
coffee 416
forest plantation 5000
forest regrowth 5000
mining 5000
other 5000
palm oil 5000
pasture 5000
rubber 5000
shifting agriculture 5000
soy 5000
urban 5000


In [338]:
subset['geometry'] = subset['geometry'].centroid

/tmp/ipykernel_1298769/595281034.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset['geometry'] = subset['geometry'].centroid


In [339]:
subset.to_file('../data/tmp/subset_5k_per_class.gpkg')
pd.DataFrame(subset).to_csv('../data/labels/dataset_v4/labels_55k.csv', index=False)

/tmp/ipykernel_1298769/595281034.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset['geometry'] = subset['geometry'].centroid


## Testing  stratified sampling of examples

In [73]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split

# Load your dataset
# Assume data is a pandas DataFrame with columns: ['latitude', 'longitude', 'class']
# Replace with the actual way you load your data
data = pd.read_csv('../data/labels/dataset_v4/labels.csv')
data['geometry'] = gpd.GeoSeries.from_wkt(data['geometry'])
data = gpd.GeoDataFrame(data)
data['geometry'] = data.geometry.centroid
data['latitude'] = data.geometry.x
data['longitude'] = data.geometry.y


# Extract coordinates and class labels
coordinates = data[['latitude', 'longitude']]
labels = data['driver']

# Cluster the data using DBSCAN
db = DBSCAN(eps=0.1, min_samples=10).fit(coordinates)  # Adjust eps and min_samples based on your data
data['cluster'] = db.labels_

# Remove noise points if any (DBSCAN labels noise as -1)
data = data[data['cluster'] != -1]

# Function to perform stratified sampling within each cluster
def stratified_sample(data, n_samples_per_class):
    samples = []
    clusters = data['cluster'].unique()
    
    for cluster in clusters:
        cluster_data = data[data['cluster'] == cluster]
        for class_label in cluster_data['driver'].unique():
            class_data = cluster_data[cluster_data['driver'] == class_label]
            if len(class_data) >= n_samples_per_class:
                sample = class_data.sample(n_samples_per_class, random_state=42)
            else:
                sample = class_data  # if not enough samples, take all
            samples.append(sample)
    
    return pd.concat(samples)

# Sample 5000 samples per class
n_samples_per_class = 5000
sampled_data = stratified_sample(data, n_samples_per_class)

# Ensure balance across classes
final_sampled_data = []
for class_label in sampled_data['driver'].unique():
    class_samples = sampled_data[sampled_data['driver'] == class_label]
    if len(class_samples) > n_samples_per_class:
        final_sampled_data.append(class_samples.sample(n_samples_per_class, random_state=42))
    else:
        final_sampled_data.append(class_samples)

final_sampled_data = pd.concat(final_sampled_data)

# Shuffle the final sampled dataset
final_sampled_data = final_sampled_data.sample(frac=1, random_state=42).reset_index(drop=True)

# Save the sampled dataset
final_sampled_data.to_csv('../data/tmp/sampled_dataset.csv', index=False)


In [76]:
final_sampled_data.to_file('../data/tmp/sampled_dataset.gpkg')

In [74]:
final_sampled_data

,geometry,driver,sampleid,dataset,latitude,longitude,cluster
0,POINT (109.32403 -0.05040),urban,100006906,worldcover.csv,109.324027,-0.050395,168
1,POINT (102.68012 2.56460),rubber,20036784,rubber.csv,102.680124,2.564600,156
2,POINT (102.10537 1.47439),urban,100007595,worldcover.csv,102.105372,1.474391,169
3,POINT (-50.40195 -24.87920),pasture,10641491,mapbiomas.csv,-50.401950,-24.879199,747
4,POINT (-48.25448 -2.42356),urban,10322019,mapbiomas.csv,-48.254484,-2.423565,297
...,...,...,...,...,...,...,...
55306,POINT (-42.09707 -22.65264),pasture,10655187,mapbiomas.csv,-42.097071,-22.652635,820
55307,POINT (17.13554 -5.19153),shifting agriculture,130001450,tyukavina_congo_basin_agri.csv,17.135538,-5.191531,1502
55308,POINT (-79.62891 -0.91170),cocoa,110002239,cocoa_ecuador.csv,-79.628906,-0.911700,968
55309,POINT (105.47272 14.76049),rubber,20010658,rubber.csv,105.472725,14.760488,127


In [79]:
data.loc[data.driver == 'palm oil'].to_file('../data/tmp/palmoil.gpkg')

## Block data split

In [5]:
df = pd.read_csv('../data/labels/dataset_v4/labels_55k.csv')
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])
gdf = gpd.GeoDataFrame(df)


In [6]:
gdf = gpd.GeoDataFrame(df)


In [7]:

coordinates = (gdf.geometry.x, gdf.geometry.y)


In [8]:
#spacing = 1
spacing = 5


block_coordinates, labels = vd.block_split(
    coordinates,
    spacing=spacing,
    #random_state=123,
)

In [9]:
blocks = np.unique(labels)

In [10]:
gdf['tmp'] = labels

In [39]:
gdf.to_file('../data/tmp/labels_55k.gpkg')

In [30]:
#Find such split that the each fold has at least n examples of all classes


finished = False
required_n_examples_per_class = 10

while not finished:
    # Shuffle the array
    np.random.shuffle(blocks)

    # Split the array into 5 equal parts
    split_arrays = np.array_split(blocks, 5)

    '''
    split 0-3 are for 4-fold cross validation (train-val)
    split 4 is test set 

    '''

    splits = []

    for item in labels:
        if item in split_arrays[0]:
            splits.append(0)
        elif item in split_arrays[1]:
            splits.append(1)
        elif item in split_arrays[2]:
            splits.append(2)
        elif item in split_arrays[3]:
            splits.append(3)
        elif item in split_arrays[4]:
            splits.append(4)


    splits = np.array(splits)
    gdf['split'] = splits
    subset = gdf

    

    min_examples_per_class = 0
    #There should be at least n examples of each class
    for i in range(5):
        classes, counts = np.unique(subset.loc[subset.split == i].driver, return_counts=True)
        if len(classes) < 12: #All classes must be present in each split 
            min_examples_per_class = 0 #Some class has 0 examples
            break
        if counts.min() > min_examples_per_class:
            min_examples_per_class = counts.min()
    
    if min_examples_per_class >= required_n_examples_per_class:
        finished = True
    

In [34]:
gdf['block'] = labels

In [36]:
for i in [0,1,2,3,4]:

    classes, counts = np.unique(subset.loc[subset.split == i].driver, return_counts=True)
    print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [ 104  146 1002  723  252 1261  437   54  780  289  174  896]
['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [ 269  112 1332  699  888  352 1031  342 1708  660  268 1244]
['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [3279   38 1010 1282 1696 1207 1626 1119 1413 1363 2108 1349]
['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [1162   38  817  943 1601  834  981  574  537 1608  335  607]
['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [ 184   82  820 1339  552 1341  916 2896  

In [84]:
classes, counts = np.unique(subset.loc[subset.split == 0].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [1787  200  946 1171  820  942 1343 1277  735  766  663 1200]


In [85]:
classes, counts = np.unique(subset.loc[subset.split == 1].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [1658   69  926  900 1384 1427 1124  831  879  993 1390 1213]


In [86]:
classes, counts = np.unique(subset.loc[subset.split == 2].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [ 225   45 1322 1009  584  968  926 1043  488 1112  852  914]


In [87]:
classes, counts = np.unique(subset.loc[subset.split == 3].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [ 132   53  768  917 1334 1018  817  867 1362 1044  972  693]


In [88]:
classes, counts = np.unique(subset.loc[subset.split == 4].driver, return_counts=True)
print(classes, counts)

['cocoa' 'coffee' 'forest plantation' 'forest regrowth' 'mining' 'other'
 'palm oil' 'pasture' 'rubber' 'shifting agriculture' 'soy' 'urban'] [1196   49 1019  989  867  640  781  967 1528 1078 1115  965]


In [47]:
labels = pd.read_csv('../data/labels/dataset_v4/labels_55k.csv')

In [48]:
labels['split_5'] = subset['split']

In [50]:
labels.to_csv('../data/labels/dataset_v4/labels_55k.csv', index=False)

In [38]:
#pd.DataFrame(subset).to_csv('../data/labels/dataset_v4/labels_55k.csv', index=False)
subset.to_file('../data/tmp/labels_tmp2.gpkg')


In [40]:
labels_tmp2 = gpd.read_file('../data/tmp/labels_tmp2.gpkg')

In [42]:
labels_tmp2.split

0        4
1        1
2        1
3        4
4        4
        ..
55298    1
55299    1
55300    2
55301    1
55302    0
Name: split, Length: 55303, dtype: int64

In [52]:
df = pd.read_csv('../data/labels/dataset_v4/labels_55k.csv')
#df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])
#gdf = gpd.GeoDataFrame(df)


In [56]:
df['split'] = labels_tmp2.split

In [58]:
for i in [1, 2, 0, 3, 4]:
    print(np.unique(df.loc[df.split == i].driver, return_counts=True))

(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
       'mining', 'other', 'palm oil', 'pasture', 'rubber',
       'shifting agriculture', 'soy', 'urban'], dtype=object), array([1658,   69,  926,  900, 1384, 1427, 1124,  831,  879,  993, 1390,
       1213]))
(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
       'mining', 'other', 'palm oil', 'pasture', 'rubber',
       'shifting agriculture', 'soy', 'urban'], dtype=object), array([ 225,   45, 1322, 1009,  584,  968,  926, 1043,  488, 1112,  852,
        914]))
(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
       'mining', 'other', 'palm oil', 'pasture', 'rubber',
       'shifting agriculture', 'soy', 'urban'], dtype=object), array([1787,  200,  946, 1171,  820,  942, 1343, 1277,  735,  766,  663,
       1200]))
(array(['cocoa', 'coffee', 'forest plantation', 'forest regrowth',
       'mining', 'other', 'palm oil', 'pasture', 'rubber',
       'shifting agriculture', 'soy', 'urba

In [61]:
df.to_csv('../data/labels/dataset_v4/labels_55k.csv', index=False)

In [59]:
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])
gdf = gpd.GeoDataFrame(df)


In [60]:
gdf.to_file('../data/tmp/tmp3.gpkg')

In [1]:
for i in [1, 2, 0, 3, 4]:
    print(np.unique(labels_tmp2.loc[labels_tmp2.split == i].driver, return_counts=True))

NameError: name 'np' is not defined

## Implementation of the block split myself

- this could be better because i can make adjustments to optimize the class ratios

Steps
- create grid
- do an intersection with the points gdf
- only keep grid cells containing points
- split cells into 5 groups
    - this can be random or I can e.g. split the cells into high-density cells and low-density cells and take this into account


In [ ]:

import geopandas as gpd
from shapely.geometry import Polygon, box
from typing import Tuple

def create_grid(square: Polygon, width: float, height: float) -> gpd.GeoDataFrame:

    minx, miny, maxx, maxy = square.bounds

    ncols = int((maxx - minx) / width)
    nrows = int((maxy - miny) / height)
    
    grid_cells = []
    for i in range(nrows):
        for j in range(ncols):
            cell_minx = minx + j * width
            cell_miny = miny + i * height
            cell_maxx = cell_minx + width
            cell_maxy = cell_miny + height
            grid_cells.append(box(cell_minx, cell_miny, cell_maxx, cell_maxy))
    
    grid_gdf = gpd.GeoDataFrame(grid_cells, columns=['geometry'])
    return grid_gdf

square_polygon = box(*gdf.total_bounds)  
cell_width, cell_height = 1, 1

grid_geodf = create_grid(square_polygon, cell_width, cell_height)

### Add country

In [38]:
df = pd.read_csv('../data/labels/dataset_v4/labels_55k.csv')
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])
gdf = gpd.GeoDataFrame(df, geometry='geometry')
world_filepath = gpd.datasets.get_path('naturalearth_lowres')
world = gpd.read_file(world_filepath)

/tmp/ipykernel_7301/1252657791.py:4: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world_filepath = gpd.datasets.get_path('naturalearth_lowres')


In [30]:
#world['geometry'] = world.to_crs(3857).geometry.buffer(100000, cap_style=3).to_crs(4326)


In [39]:
gdf['country'] = gdf.geometry.intersects(world).name

/tmp/ipykernel_7301/291178877.py:1: UserWarning: The indices of the two GeoSeries are different.
  gdf['country'] = gdf.geometry.intersects(world).name
/tmp/ipykernel_7301/291178877.py:1: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: None
Right CRS: EPSG:4326

  gdf['country'] = gdf.geometry.intersects(world).name


In [40]:
gdf.crs = 4326

In [41]:
#This results in over 1.6k missing values because the world map is low resolution so many points don't intersect with any country
#gdf_with_countries = gdf.sjoin(world[['name', 'geometry']], how="left")
#gdf_with_countries['country'] = gdf_with_countries.name
#gdf_with_countries = gdf_with_countries.drop(['index_right', 'name'], axis=1)

In [64]:
from shapely.ops import nearest_points


def find_intersections_or_nearest(gdf_points, gdf_polygons):
    """
    For each point in gdf_points, find which polygon it intersects with. 
    If it doesn't intersect any polygon, find the closest polygon.
    
    Parameters:
    gdf_points (GeoDataFrame): A GeoDataFrame containing points.
    gdf_polygons (GeoDataFrame): A GeoDataFrame containing polygons.
    
    Returns:
    GeoDataFrame: A GeoDataFrame of points with a new column 'nearest_or_intersecting_polygon' indicating the index of the intersecting or nearest polygon.
    """
    # Initialize a column to store the index of the nearest or intersecting polygon
    gdf_points['country'] = None
    
    for point_index, point_row in gdf_points.iterrows():
        # Check for intersections first
        intersects = gdf_polygons.intersects(point_row.geometry)
        if intersects.any():
            # If there's an intersection, use the first intersecting polygon
            idx = intersects.idxmax()
            gdf_points.at[point_index, 'country'] = gdf_polygons.iloc[idx]['name']
        else:
            # Find the nearest polygon if no intersection
            # Calculate the nearest points between the current point and all polygons, then find the minimum
            distances = gdf_polygons.geometry.apply(lambda poly: point_row.geometry.distance(nearest_points(point_row.geometry, poly)[1]))
            nearest_polygon_index = distances.idxmin()
            gdf_points.at[point_index, 'country'] = gdf_polygons.iloc[nearest_polygon_index]['name']
    
    return gdf_points

In [70]:
country_name_mapping = {
    "Vietnam" : "Viet Nam",
    "Tanzania" : "United Republic of Tanzania",
    "Dem. Rep. Congo" : "Democratic Republic of the Congo",
    "Dominican Rep." : "Dominican Republic",
    "Bolivia" : "Bolivia (Plurinational State of)",
    "Venezuela" : "Venezuela (Bolivarian Republic of)",
    "Central African Rep.": "Central African Republic",
    "Eq. Guinea" : "Equatorial Guinea",
    "eSwatini" : "Eswatini",
    "Laos" : "Lao People's Democratic Republic",
    "Taiwan" : "China, Taiwan Province of",
    "Brunei" : "Brunei Darussalam",
    "S. Sudan" : "South Sudan",
    "Solomon Is." : "Solomon Islands"
}

names_to_edit = [key for key in country_name_mapping.keys()]
for name in names_to_edit:
    gdf_with_countries.loc[gdf_with_countries['country'] == name, 'country'] = country_name_mapping[name]
    

In [66]:
gdf_with_countries = find_intersections_or_nearest(gdf, world)

In [73]:
pd.DataFrame(gdf_with_countries).to_csv('../data/labels/dataset_v4/labels_55k.csv', index=False)

### Just querying the data

In [4]:
df = pd.read_csv('../data/labels/dataset_v4/labels_55k.csv')
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])
gdf = gpd.GeoDataFrame(df, geometry='geometry')

In [8]:
np.unique(gdf.loc[gdf.country.isin(["Malaysia", "Ecuador"])].driver, return_counts=True)


(array(['cocoa', 'forest regrowth', 'mining', 'palm oil', 'rubber',
        'urban'], dtype=object),
 array([1683,  184,   39,  720,  249,  689]))

In [9]:
np.unique(gdf.loc[gdf.country == 'Colombia'].driver, return_counts=True)

(array(['cocoa', 'forest regrowth', 'mining', 'palm oil', 'urban'],
       dtype=object),
 array([  3, 246,  59,  24,  56]))

In [13]:
np.unique(gdf.loc[gdf.country.isin(["Cameroon", "Ecuador", "Paraguay", "Malaysia", "Thailand"])].driver, return_counts=True)

(array(['cocoa', 'forest regrowth', 'mining', 'palm oil', 'pasture',
        'rubber', 'shifting agriculture', 'soy', 'urban'], dtype=object),
 array([1683,  199,   44,  767,    1, 2571,  358,    1, 1101]))

In [13]:
np.unique(gdf.loc[gdf.country == 'Malaysia'].driver, return_counts=True)

(array(['forest regrowth', 'mining', 'palm oil', 'rubber', 'urban'],
       dtype=object),
 array([ 99,  24, 703, 249, 644]))

In [9]:
np.unique(gdf.loc[gdf.country == "Côte d'Ivoire"].driver, return_counts=True)

(array(['cocoa', 'forest regrowth', 'mining', 'palm oil', 'urban'],
       dtype=object),
 array([1551,    1,   16,   39,   25]))